In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"
import bayesflow as bf
import numpy as np
np.set_printoptions(suppress=True)
from numba import njit, prange
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
@njit
def softplus(x):
    return np.log1p(np.exp(x))

## Priors

In [ ]:
PARAM_NAMES = [
    r"$mu_{\text{intercept}}$", r"$mu_{\text{theta}}$",
    r"$mu_{\text{slope}}$", r"$mu_{\text{var}}$",
    r"$a_{\text{intercept}}$", r"$a_{\text{slope}}$",
    r"$\lambda$", r"$\tau$", r"$\tau_{\text{var}}$"
]

In [ ]:
@njit
def sample_cdm_prior() -> np.ndarray:
    mu_intercept = np.random.normal(1, 2)
    mu_theta = 2.0*np.pi * (np.random.beta(3.0, 3.0) - 0.5)
    mu_slope = np.random.normal(0, 2)
    mu_var = np.random.gamma(1, 0.2)
    a_intercept = np.random.gamma(10.0, 0.3)
    a_slope = np.random.normal(0.0, 1.0)
    lamda = np.random.gamma(1, 0.4)
    tau = np.random.gamma(3.0, 0.2)
    tau_var = np.random.uniform(0, tau*2)
    return np.array(
        [
            mu_intercept, mu_theta, mu_slope, mu_var,
            a_intercept, a_slope, lamda, tau, tau_var
        ]
    )

## Design Matrix

In [ ]:
@njit
def sample_design_mats(num_trials: int) -> np.ndarray:
    mat_1 = np.random.random(num_trials)
    mat_2 = np.random.random(num_trials)
    return np.stack((mat_1, mat_2), axis=1)

## Simulator

In [ ]:
@njit
def sample_cdm_trial(
    mu: np.ndarray,
    a: float,
    lamda: float,
    tau: float,
    dt: float = 0.001,
    s: float = 1.0,
    max_iters: int = int(1e5),
) -> np.ndarray:
    c = np.sqrt(dt) * s
    # exponentially collapsing threshold
    t = np.arange(0, max_iters * dt, dt)
    threshold = a * np.exp(-lamda * t)
    x = np.zeros(2)
    for i_iter in range(max_iters):
        x += mu*dt + c * np.random.randn(2)
        if np.linalg.norm(x, 2) >= threshold[i_iter]:
            return np.array([tau + i_iter * dt, np.arctan2(x[1], x[0])])
    # No decision within max_iters
    return np.array([-1.0, -1.0])

In [ ]:
@njit
def sample_generative_model(
    batch_size: int = 32,
    num_trials: int = 200,
    dt: float = 0.001,
    s: float = 1.0,
    max_iters: int = int(1e5)
) -> tuple:
    prior_draws = np.empty((batch_size, 9), dtype=np.float32)
    design_mats = np.empty((batch_size, num_trials, 2), dtype=np.float32)
    data = np.empty((batch_size, num_trials, 2), dtype=np.float32)
    for batch in range(batch_size):
        prior_draws[batch] = sample_cdm_prior()
        design_mats[batch] = sample_design_mats(num_trials)
        x_v = design_mats[batch, :, 0]
        x_a = design_mats[batch, :, 1]
        # context dependent drift length
        drift_length = softplus(prior_draws[batch, 0] + prior_draws[batch, 2] * x_v)
        mu = drift_length[:, None] * np.array(
            [np.cos(prior_draws[batch, 1]), np.sin(prior_draws[batch, 1])]
        )
        # drift inter-trial variability
        noise = np.random.normal(0.0, prior_draws[batch, 3], size=mu.shape)
        mu = mu + noise
        # context dependent threshold
        a = softplus(prior_draws[batch, 4] + prior_draws[batch, 5] * x_a)
        ndt = np.random.uniform(
            prior_draws[batch, 7] - prior_draws[batch, 8] / 2,
            prior_draws[batch, 7] + prior_draws[batch, 8] / 2,
            size=num_trials
        )
        for i in range(num_trials):
            data[batch, i] = sample_cdm_trial(
                mu[i, :], a[i], prior_draws[batch, 6],
                ndt[i], dt, s, max_iters
        )
    return prior_draws, design_mats, data


In [ ]:
class SimulationCDM:
    def __init__(
        self,
        num_trials: int = 200,
        dt: float = 0.001,
        s: float = 1.0,
        max_iters: int = int(1e5)
    ):
        self.num_trials = num_trials
        self.dt = dt
        self.s = s
        self.max_iters = max_iters

    def sample(self, batch_shape) -> dict:
        batch_size = batch_shape[0]
        prior_draws, design_mats, data = sample_generative_model(
            batch_size,
            self.num_trials,
            self.dt,
            self.s,
            self.max_iters
        )
        return dict(
            prior_draws=prior_draws,
            design_mats=design_mats,
            data=data
        )

In [ ]:
simulator = SimulationCDM()

In [ ]:
%%time
sim_data = simulator.sample((32, ))

In [ ]:
n_sims = sim_data['data'][:8].shape[0]
n_cols = 4
n_rows = 2
fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(3*n_cols, 2.8*n_rows)
)
axes = axes.flatten()
hist_color = "#4C72B0"
for i in range(n_sims):
    rts = sim_data['data'][i, :, 0]
    ax = axes[i]
    sns.histplot(
        rts, bins=20, stat="density", kde=True,
        color=hist_color, alpha=0.6, edgecolor="none", ax=ax
    )
    ax.set_title(f"Sim {i+1}", fontsize=12)
    ax.tick_params(axis="both", labelsize=10)
sns.despine()
plt.tight_layout(rect=[0, 0, 1, 0.97])

In [ ]:
fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(3*n_cols, 2.8*n_rows)
)
axes = axes.flatten()
hist_color = "#4C72B0"
for i in range(n_sims):
    rts = sim_data['data'][i, :, 1]
    ax = axes[i]
    sns.histplot(
        rts, bins=20, stat="density", kde=True,
        color=hist_color, alpha=0.6, edgecolor="none", ax=ax
    )
    ax.set_title(f"Sim {i+1}", fontsize=12)
    ax.tick_params(axis="both", labelsize=10)
sns.despine()
plt.tight_layout(rect=[0, 0, 1, 0.97])

## Workflown

In [ ]:
adapter = (
    bf.Adapter()
    .convert_dtype("float64", "float32")
    .rename("prior_draws", "inference_variables")
    .concatenate(["data", "design_mats"], into="summary_variables")
)

In [ ]:
batch_size = 32
epochs = 100
summary_network = bf.networks.SetTransformer(summary_dim=16, seed_dim=128)
inference_network = bf.networks.FlowMatching()

## Offline Training

In [ ]:
NUM_TEST_SIMS = 10000
NUM_VALIDATION_SIMS = 200

In [ ]:
offline_sims = simulator.sample((NUM_TEST_SIMS, ))
validation_sims = simulator.sample((NUM_VALIDATION_SIMS, ))

In [ ]:
workflow = bf.BasicWorkflow(
    inference_network=inference_network,
    summary_network=summary_network,
    simulator=simulator,
    adapter=adapter,
    checkpoint_filepath=f"../checkpoints/offline_training_cdm"
)

## Evaluation

In [ ]:
NUM_VALIDATION_SIMS = 500
validation_sims = simulator.sample((NUM_VALIDATION_SIMS, ))

In [ ]:
fig = workflow.plot_default_diagnostics(
    test_data=validation_sims,
    variable_names=PARAM_NAMES,
    num_samples=500,
    loss_kwargs={"figsize": (16, 3), "label_fontsize": 12},
    recovery_kwargs={"figsize": (16, 6), "label_fontsize": 12},
    calibration_ecdf_kwargs={"figsize": (16, 6), "legend_fontsize": 8, "difference": True, "label_fontsize": 12},
    z_score_contraction_kwargs={"figsize": (16, 6), "label_fontsize": 12}
)